# TARDIS — Predictive Modelling (V1 Baseline)
### *Forecasting SNCF Train Delays with Machine Learning*

---

## Context & Objective
This notebook represents the first iteration (V1 Baseline) of the predictive modeling pipeline for the TARDIS project. Following the initial guidelines, our objective is to build a machine learning regression model to forecast the continuous target variable: `Average delay of all trains at arrival`.

At this stage, we are intentionally feeding the model with our unrefined baseline dataset (`cleaned_dataset.csv` V1) to measure how data anomalies and typos impact machine learning performance, establishing a starting benchmark.

## 1. Environment Setup & Data Loading

In this section, we import the required machine learning libraries from `scikit-learn` and load our baseline dataset from Google Drive.

In [11]:
# Cell 1: Environment Setup and Loading Data
import os
import pandas as pd
import numpy as np
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Mount Google Drive for persistent file access
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

# Define paths matching the project directory structure
FOLDER = "/content/drive/MyDrive/TARDIS-PROJECT"
DATA_PATH = os.path.join(FOLDER, "cleaned_dataset.csv")

# Load the unrefined V1 baseline dataset
df = pd.read_csv(DATA_PATH)
print(f"✅ Baseline dataset loaded successfully: {len(df):,} rows.")

✅ Baseline dataset loaded successfully: 8,211 rows.


## 2. Data Preprocessing & Feature Selection

Machine learning algorithms require purely numerical matrix inputs. In this baseline setup, we select our core operational features (`Year`, `Month`, `Departure station`, `Arrival station`) and convert the categorical text columns using One-Hot Encoding via pandas dummies.

In [12]:
# Cell 2: Feature Selection and Categorical Encoding
# Select features (X) and the continuous target variable (y)
feature_cols = ["Year", "Month", "Departure station", "Arrival station"]
target_col = "Average delay of all trains at arrival"

# Drop any residual rows missing crucial modeling features or targets
df_model = df[feature_cols + [target_col]].dropna()

X = df_model[feature_cols]
y = df_model[target_col]

# Convert categorical text columns into dummy indicator variables (One-Hot Encoding)
X_encoded = pd.get_dummies(
    X, columns=["Departure station", "Arrival station"], drop_first=True
)
print(
    f"📊 Encoded features matrix shape: {X_encoded.shape[0]} rows × {X_encoded.shape[1]} columns."
)

📊 Encoded features matrix shape: 8211 rows × 118 columns.


## 3. Naive Baseline Establishment

To fulfill the project requirements, we must establish a naive baseline performance before training our model. This baseline represents a simple, non-intelligent strategy that always predicts the historical mean delay value for every single trip.

In [13]:
# Cell 3: Train-Test Split and Model Fitting
# Split the data into training and validation subsets
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

# Initialize and fit a baseline Decision Tree Regressor to establish our starting performance
model = DecisionTreeRegressor(max_depth=10, random_state=42)
model.fit(X_train, y_train)

print("🎯 Baseline model training completed successfully.")

🎯 Baseline model training completed successfully.


## 4. Baseline Model Evaluation
We compute standard regression metrics to quantify model performance: Root Mean Squared Error (RMSE), Mean Absolute Error (MAE), and the R² score.

In [14]:
# Cell 3: Establish Naive Baseline Prediction (Mean Value)
# Create a naive prediction array filled entirely with the historical mean delay value
y_mean_baseline = np.full(shape=y.shape, fill_value=y.mean())

# Compute baseline reference error metrics
baseline_rmse = np.sqrt(mean_squared_error(y, y_mean_baseline))
baseline_mae = mean_absolute_error(y, y_mean_baseline)
baseline_r2 = r2_score(y, y_mean_baseline)

print("── Naive Mean Baseline Performance ──")
print(f"   Baseline RMSE: {baseline_rmse:.3f} minutes")
print(f"   Baseline MAE : {baseline_mae:.3f} minutes")
print(f"   Baseline R²  : {baseline_r2:.3f}")

── Naive Mean Baseline Performance ──
   Baseline RMSE: 12.411 minutes
   Baseline MAE : 6.548 minutes
   Baseline R²  : 0.000


## 4. Train-Test Split & Model Training

We split our encoded features and target arrays into a training set (80%) to fit our machine learning algorithm and a testing set (20%) to evaluate its capacity to generalize on unseen historical records. We initiate our journey with a standard Decision Tree Regressor.

In [15]:
# Cell 4: Train-Test Split and Model Fitting
# Split the data into training (80%) and testing (20%) subsets
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

# Initialize and fit a baseline Decision Tree Regressor
model = DecisionTreeRegressor(max_depth=10, random_state=42)
model.fit(X_train, y_train)

print("🎯 Machine learning model training completed successfully.")

🎯 Machine learning model training completed successfully.


## 5. Model Performance Evaluation

We compute the performance metrics for our trained Decision Tree model on the test data to verify if it successfully outperforms our naive mean baseline strategy.

In [16]:
# Cell 5: Model Performance Evaluation Metrics
# Generate predictions on the unseen test partition
y_pred = model.predict(X_test)

# Calculate model error metrics
model_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
model_mae = mean_absolute_error(y_test, y_pred)
model_r2 = r2_score(y_test, y_pred)

print("── Trained Model Performance Metrics ──")
print(f"   Model RMSE: {model_rmse:.3f} minutes")
print(f"   Model MAE : {model_mae:.3f} minutes")
print(f"   Model R²  : {model_r2:.3f}")

── Trained Model Performance Metrics ──
   Model RMSE: 13.788 minutes
   Model MAE : 6.736 minutes
   Model R²  : -0.164


## 6. Model Serialization & Export

To complete the modeling step, we serialize our trained machine learning object and export it as a binary file (`model.pkl`) so it can be integrated into our interactive Streamlit application.

In [17]:
# Cell 6: Export and Serialize Trained Model Object
import joblib

# Ensure FOLDER and MODEL_PATH are available even if cells are run out of order
if "FOLDER" not in locals():
    FOLDER = "/content/drive/MyDrive/TARDIS-PROJECT"

MODEL_PATH = os.path.join(FOLDER, "model.pkl")

# Save the final trained model object to Google Drive for dashboard deployment
joblib.dump(model, MODEL_PATH)
print(f"💾 Trained model successfully exported to: {MODEL_PATH}")

💾 Trained model successfully exported to: /content/drive/MyDrive/TARDIS-PROJECT/model.pkl


## 7. Justification & Iteration Analysis

In this first baseline version, the machine learning model performance shows a very low R² score and an error margin close to the naive prediction strategy.

This behavior is expected and justifies our next operational step: our data input is currently corrupted by duplicate stations due to spelling typos (e.g., `PAOIS` vs `PARIS`) and extreme uncleaned sentinel values. To fix this, we need to return to `tardis_eda.ipynb` and deploy our Version 2 automated Fuzzy Matching cleaning pipeline to feed the model high-quality structured data.

---

# Part 2: Advanced Predictive Modelling (V2 Expert)

## Dataset Evolution & Model Upgrade
After executing our advanced V2 cleaning pipeline in `tardis_eda.ipynb`, the target distribution changed dramatically because extreme system sentinel values were removed.

While the baseline Decision Tree performance dropped due to the loss of artificial patterns (overfitting on noise), we will now upgrade our modeling strategy. We will train a robust **Random Forest Regressor** to properly capture the clean, complex operational realities of the train network.

### .1 Loading Refined Dataset & Encoding
We load the updated `cleaned_dataset.csv` generated by our V2 pipeline, extract the features, and perform categorical encoding.

In [18]:
# Cell 7: Advanced Data Preparation (V2 Optimized)
# This cell loads the high-quality V2 dataset and prepares features including dense continuous variables.
import os
import pandas as pd

FOLDER = "/content/drive/MyDrive/TARDIS-PROJECT"
DATA_PATH = os.path.join(FOLDER, "cleaned_dataset.csv")

# Load the freshly updated expert dataset
df_v2 = pd.read_csv(DATA_PATH)

# NEW: We expand the feature set by adding numerical operational context
feature_cols_v2 = [
    "Year",
    "Month",
    "Departure station",
    "Arrival station",
    "Season",
    "Number of scheduled trains",  # Traffic volume
    "Cancellation rate",  # Line health indicator
    "Average journey time",  # Route distance
]
target_col = "Average delay of all trains at arrival"

# Drop rows missing any of our critical operational metrics
df_model_v2 = df_v2[feature_cols_v2 + [target_col]].dropna()

X_v2 = df_model_v2[feature_cols_v2]
y_v2 = df_model_v2[target_col]

# One-Hot Encode all categorical attributes safely
X_encoded_v2 = pd.get_dummies(
    X_v2,
    columns=["Departure station", "Arrival station", "Season"],
    drop_first=True,
)
print(
    f"📊 Optimized V2 Features matrix shape: {X_encoded_v2.shape[0]} rows × {X_encoded_v2.shape[1]} columns."
)

📊 Optimized V2 Features matrix shape: 5441 rows × 124 columns.


### .2 Training an Ensemble Model (Random Forest)
We split our clean data (80/20) and deploy a Random Forest Regressor. This ensemble method aggregates multiple decision trees to stabilize variance and significantly increase predictive accuracy.

In [19]:
# Cell 8: Random Forest Model Training (Optimized)
# This cell splits the enriched data and trains a tuned Random Forest Regressor.
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# Split into training and testing partitions
X_train_v2, X_test_v2, y_train_v2, y_test_v2 = train_test_split(
    X_encoded_v2, y_v2, test_size=0.2, random_state=42
)

# NEW: Tuned hyperparameters to capture dense continuous patterns
model_v2 = RandomForestRegressor(
    n_estimators=150, max_depth=15, min_samples_split=5, random_state=42, n_jobs=-1
)
model_v2.fit(X_train_v2, y_train_v2)

print("🌲 Tuned Random Forest model trained successfully on enriched data.")

🌲 Tuned Random Forest model trained successfully on enriched data.


### .3 Performance Evaluation & Model Export
We evaluate our expert model against the test subset and save the final serialized object to run our interactive Streamlit application.

In [20]:
# Cell 9: Expert Model Evaluation and Serialization
# This cell computes final regression metrics for the V2 model and overwrites model.pkl.
import os
import joblib
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Generate predictions
y_pred_v2 = model_v2.predict(X_test_v2)

# Calculate final metrics
v2_rmse = np.sqrt(mean_squared_error(y_test_v2, y_pred_v2))
v2_mae = mean_absolute_error(y_test_v2, y_pred_v2)
v2_r2 = r2_score(y_test_v2, y_pred_v2)

print("── Trained V2 Model Performance Metrics ──")
print(f"   Model RMSE: {v2_rmse:.3f} minutes")
print(f"   Model MAE : {v2_mae:.3f} minutes")
print(f"   Model R²  : {v2_r2:.3f}")

# Overwrite model.pkl with our high-quality model
MODEL_PATH = os.path.join(FOLDER, "model.pkl")
joblib.dump(model_v2, MODEL_PATH)
print(f"\n💾 Production model successfully exported to: {MODEL_PATH}")

── Trained V2 Model Performance Metrics ──
   Model RMSE: 11.713 minutes
   Model MAE : 5.976 minutes
   Model R²  : -0.008

💾 Production model successfully exported to: /content/drive/MyDrive/TARDIS-PROJECT/model.pkl
